# Лабораторная работа 1. Метод ближайших соседей

    ## Данные и протокол

    - **Классификация.** [Optical Recognition of Handwritten Digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html), 1797 изображений размером 8×8, классы от 0 до 9. Практическая задача — распознавание цифр в оцифрованных рукописных формах. Используется поставляемая в sklearn часть UCI Optdigits.
    - **Регрессия.** [Diabetes (Efron и соавт.)](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html), 442 наблюдения и 10 исходных признаков. Цель — численный показатель прогрессирования болезни через год после исходного обследования. Это учебный эксперимент на исторической выборке.

    Для классификации основная метрика **macro F1**, она одинаково взвешивает каждый класс. Accuracy показывает долю верных ответов. Для регрессии основная метрика **MAE** выражает типичную абсолютную ошибку в единицах цели. RMSE сильнее реагирует на крупные промахи. R² показывает качество относительно прогноза средним.

    Разбиение 80/20 с seed 42 одинаково в пяти работах. Для классификации используется стратификация. Гиперпараметры выбираем по среднему результату 5-fold CV на обучающей части. Масштабирование, где оно используется, обучаем заново внутри каждого фолда и на полной обучающей выборке перед оценкой на тесте.

In [1]:
# Импортируем зависимости, задаём разбиение и функции расчёта метрик.
from hashlib import sha256
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp, softmax
from sklearn.datasets import load_diabetes, load_digits
from sklearn.ensemble import (GradientBoostingClassifier, GradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from threadpoolctl import threadpool_limits

SEED = 42

def load_task(classification):
    data = load_digits() if classification else load_diabetes(scaled=False)
    ids = np.arange(len(data.target))
    train, test = train_test_split(ids, test_size=0.2, random_state=SEED,
                                   stratify=data.target if classification else None)
    return data, train, test

def partition(data, train, test, scale):
    X_train, X_test = data.data[train], data.data[test]
    if scale:
        scaler = StandardScaler().fit(X_train)
        X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
    return X_train, X_test

def metrics(y, prediction, classification):
    if classification:
        return {"macro_f1": f1_score(y, prediction, average="macro", zero_division=0),
                "accuracy": accuracy_score(y, prediction)}
    return {"mae": mean_absolute_error(y, prediction),
            "rmse": mean_squared_error(y, prediction) ** 0.5,
            "r2": r2_score(y, prediction)}

In [2]:
# Проверяем размеры и распределение обучающих данных.
for classification in (True, False):
    data, train, test = load_task(classification)
    print("Классификация" if classification else "Регрессия")
    print(f"Обучение: {len(train)}, тест: {len(test)}, признаков: {data.data.shape[1]}")
    print("Пропуски в обучении:", np.isnan(data.data[train]).sum())
    print(np.bincount(data.target[train]) if classification
          else pd.Series(data.target[train]).describe().round(2).to_string())

Классификация
Обучение: 1437, тест: 360, признаков: 64
Пропуски в обучении: 0
[142 146 142 146 145 145 145 143 139 144]
Регрессия
Обучение: 353, тест: 89, признаков: 10
Пропуски в обучении: 0
count    353.00
mean     153.74
std       78.06
min       25.00
25%       86.00
50%      142.00
75%      214.00
max      346.00


## Бейзлайн sklearn

Фиксируем исходную конфигурацию и измеряем её качество на отложенной части.

In [3]:
# Задаём исходные параметры sklearn и функцию оценки модели.
LAB = 1
def make_model(classification, own, params):
    params = params.copy()
    params.pop("scale")
    if LAB == 1:
        if own:
            return OwnKNN(classification, **params)
        return (KNeighborsClassifier if classification else KNeighborsRegressor)(**params)
    if LAB == 2:
        strength = params["strength"]
        if own:
            return OwnLinear(classification, strength)
        if classification:
            return LogisticRegression(C=strength, max_iter=2500, tol=1e-7)
        return Ridge(alpha=strength) if strength else LinearRegression()
    if LAB == 3:
        if own:
            return OwnTree(classification, **params)
        return (DecisionTreeClassifier if classification else DecisionTreeRegressor)(
            random_state=SEED, **params)
    if LAB == 4:
        if own:
            return OwnForest(classification, **params)
        return (RandomForestClassifier if classification else RandomForestRegressor)(
            random_state=SEED, n_jobs=1, **params)
    if own:
        return OwnBoost(classification, **params)
    return (GradientBoostingClassifier if classification else GradientBoostingRegressor)(
        random_state=SEED, **params)

def evaluate(data, train, test, classification, params, own=False):
    X_train, X_test = partition(data, train, test, params["scale"])
    model = make_model(classification, own, params)
    with threadpool_limits(limits=1):
        model.fit(X_train, data.target[train])
        prediction = model.predict(X_test)
    return metrics(data.target[test], prediction, classification)

def validate_candidates(data, train, classification, candidates):
    splitter = (StratifiedKFold if classification else KFold)(
        n_splits=5, shuffle=True, random_state=SEED)
    folds = list(splitter.split(train, data.target[train]))
    metric = "macro_f1" if classification else "mae"
    rows = []
    for hypothesis, params in candidates:
        scores = [evaluate(data, train[fit], train[val], classification, params)[metric]
                  for fit, val in folds]
        rows.append({"гипотеза": hypothesis, "mean": np.mean(scores), "std": np.std(scores)})
    table = pd.DataFrame(rows).set_index("гипотеза")
    winner = table["mean"].idxmax() if classification else table["mean"].idxmin()
    return table, winner, dict(candidates)[winner]

In [4]:
# Обучаем базовые библиотечные модели и считаем метрики.
bases = {}
baseline_scores = {}
for classification in (True, False):
    params = {'scale': False, **{'n_neighbors': 5, 'p': 2, 'weights': 'uniform'}}
    if LAB == 2:
        params['strength'] = 1.0 if classification else 0.0
    if LAB == 4:
        params['max_features'] = 'sqrt' if classification else None
    bases[classification] = params
    data, train, test = load_task(classification)
    baseline_scores[classification] = evaluate(data, train, test, classification, params)
    print(('Классификация' if classification else 'Регрессия'), baseline_scores[classification])

Классификация {'macro_f1': 0.9830456008969376, 'accuracy': 0.9833333333333333}
Регрессия {'mae': 55.57078651685392, 'rmse': 66.88952152533041, 'r2': 0.15551530000243063}


## Проверка гипотез и улучшенный бейзлайн

Масштабирование может улучшить сравнение объектов на разномасштабных признаках. Меняя число соседей, расстояние и веса, проверяем компромисс между локальностью и сглаживанием.

Исходная конфигурация тоже участвует в CV. Если улучшения не подтверждаются, сохраняем исходные параметры.

In [5]:
# Сравниваем конфигурации на обучающих фолдах, затем оцениваем победителя на тесте.
chosen = {}
improved_scores = {}
for classification in (True, False):
    base = bases[classification]
    changes = [('Масштабирование', {'scale': True}), ('Меньше соседей с весом 1/d', {'n_neighbors': 3, 'weights': 'distance'}), ('Манхэттенское расстояние с масштабированием', {'scale': True, 'p': 1, 'weights': 'distance'}), ('Больше соседей с масштабированием', {'scale': True, 'n_neighbors': 15, 'weights': 'distance'})]
    if LAB == 2:
        strengths = [0.01, 0.1, 10.0] if classification else [0.1, 1.0, 10.0, 50.0]
        changes = [('Масштабирование', {'scale': True})]
        changes += [(f'Масштабирование и штраф {v}', {'scale': True, 'strength': v})
                    for v in strengths]
    if LAB == 4:
        alternative = None if classification else 'sqrt'
        changes = [*changes, ('Другой размер подмножества признаков', {'max_features': alternative})]
    candidates = [('Бейзлайн', base)] + [(name, base | patch) for name, patch in changes]
    data, train, test = load_task(classification)
    table, winner, chosen[classification] = validate_candidates(data, train, classification, candidates)
    improved_scores[classification] = evaluate(data, train, test, classification, chosen[classification])
    print('Классификация' if classification else 'Регрессия')
    print(table.round(4).to_string())
    print('Выбор по CV:', winner, chosen[classification])
    print('Тест:', improved_scores[classification])

Классификация
                                               mean     std
гипотеза                                                   
Бейзлайн                                     0.9875  0.0074
Масштабирование                              0.9756  0.0098
Меньше соседей с весом 1/d                   0.9868  0.0050
Манхэттенское расстояние с масштабированием  0.9692  0.0116
Больше соседей с масштабированием            0.9629  0.0073
Выбор по CV: Бейзлайн {'scale': False, 'n_neighbors': 5, 'p': 2, 'weights': 'uniform'}
Тест: {'macro_f1': 0.9830456008969376, 'accuracy': 0.9833333333333333}
Регрессия
                                                mean     std
гипотеза                                                    
Бейзлайн                                     55.7826  6.3336
Масштабирование                              49.4423  4.0527
Меньше соседей с весом 1/d                   58.2587  5.6487
Манхэттенское расстояние с масштабированием  50.0549  4.6185
Больше соседей с масштабирование

## Собственная реализация

Ближайших соседей ищем по расстоянию Минковского. Класс определяет взвешенное голосование, регрессионное значение — взвешенное среднее.

In [6]:
# Реализуем модель и предсказание средствами NumPy и SciPy.
class OwnKNN:
    def __init__(self, classification, n_neighbors=5, p=2, weights="uniform"):
        self.classification = classification
        self.n_neighbors = n_neighbors
        self.p = p
        self.weights = weights

    def fit(self, X, y):
        self.X = np.asarray(X, dtype=float)
        self.y = np.asarray(y)
        self.classes = np.unique(y)
        return self

    def predict(self, X):
        predictions = []
        for row in X:
            delta = np.abs(self.X - row)
            distances = np.sum(delta ** self.p, axis=1) ** (1 / self.p)
            # При одинаковом расстоянии сохраняем исходный порядок объектов.
            idx = np.argsort(distances, kind="stable")[:self.n_neighbors]
            d, labels = distances[idx], self.y[idx]
            weights = np.ones(len(idx))
            if self.weights == "distance":
                weights = (d == 0).astype(float) if np.any(d == 0) else 1 / d
            if self.classification:
                totals = [weights[labels == label].sum() for label in self.classes]
                predictions.append(self.classes[np.argmax(totals)])
            else:
                predictions.append(np.average(labels, weights=weights))
        return np.array(predictions)

In [7]:
# Сравниваем четыре варианта на одном тесте и сохраняем метрики для итогового ноутбука.
records = []
for classification in (True, False):
    data, train, test = load_task(classification)
    own_base = evaluate(data, train, test, classification, bases[classification], own=True)
    own_improved = evaluate(data, train, test, classification, chosen[classification], own=True)
    scores = {
        "sklearn_base": baseline_scores[classification],
        "sklearn_tuned": improved_scores[classification],
        "own_base": own_base,
        "own_tuned": own_improved,
    }
    task = "classification" if classification else "regression"
    print(task, pd.DataFrame(scores).T.round(4).to_string(), sep="\n")
    split = sha256(np.asarray(test, dtype="<i8").tobytes()).hexdigest()[:16]
    for version, values in scores.items():
        records.append(dict(lab=LAB, task=task, version=version, n_test=len(test), split=split, **values))

result_path = Path("results") / f"lab-{LAB}.json"
result_path.parent.mkdir(exist_ok=True)
result_path.write_text(json.dumps(records, indent=2) + "\n", encoding="utf-8")

classification
               macro_f1  accuracy
sklearn_base      0.983    0.9833
sklearn_tuned     0.983    0.9833
own_base          0.983    0.9833
own_tuned         0.983    0.9833
regression
                   mae     rmse      r2
sklearn_base   55.5708  66.8895  0.1555
sklearn_tuned  44.5765  54.6463  0.4364
own_base       55.5708  66.8895  0.1555
own_tuned      44.5765  54.6463  0.4364


## Выводы

**Классификация.** MACRO_F1 библиотечной модели не изменилось на тесте (0.9830 → 0.9830). Собственная реализация дала 0.9830 с базовыми параметрами и 0.9830 с настройками, выбранными по CV для sklearn. Разность между собственным и библиотечным вариантами после настройки составила +0.0000.

**Регрессия.** MAE библиотечной модели улучшилось на тесте (55.5708 → 44.5765). Собственная реализация дала 55.5708 с базовыми параметрами и 44.5765 с настройками, выбранными по CV для sklearn. Разность между собственным и библиотечным вариантами после настройки составила +0.0000.

Результаты относятся к одному отложенному разбиению. Выбранные по CV настройки не гарантируют улучшения на тесте.